In [ ]:
%run "./Setup.ipynb"

In [ ]:
spark

In [ ]:
inputPath = "E:\\PySpark\\data\\users.json"

In [ ]:
from pyspark.sql.functions import *

In [ ]:
df1 = spark.read.json(inputPath)
df1.show()

In [ ]:
qry = """select age, count(1) as count
        from users
        where age is not null
        group by age
        order by count
        limit 5"""

### Local Temp View

In [ ]:
df1.createOrReplaceTempView("users")

In [ ]:
spark.catalog.listTables()

In [ ]:
spark.sql("select * from users where age is not null and phone is not null").show()

In [ ]:
spark2 = spark.newSession()

In [ ]:
## this does not work
spark2.sql("select * from users where age is not null and phone is not null").show()

In [ ]:
df2 = spark2.read.json(inputPath)
df2.show()

In [ ]:
df2.createOrReplaceTempView("users2")

In [ ]:
spark2.catalog.listTables()

In [ ]:
spark.catalog.listTables()

In [ ]:
spark2.sql("select * from users2 where age is not null and phone is not null").show()

In [ ]:
## this does not work
spark.sql("select * from users2 where age is not null and phone is not null").show()

### Global Temp View

In [ ]:
df1.createOrReplaceGlobalTempView("gusers")

In [ ]:
spark.catalog.currentDatabase()

In [ ]:
spark.catalog.listTables("global_temp")

In [ ]:
spark.sql("select * from global_temp.gusers where age > 15").show()

In [ ]:
spark2.sql("select * from global_temp.gusers where age > 15").show()

#### Deleting temp views

In [ ]:
spark.catalog.listTables("global_temp")

In [ ]:
spark.catalog.dropTempView("users")

In [ ]:
spark.catalog.dropGlobalTempView("gusers")

### Managed Tables

In [ ]:
spark.conf.get("spark.sql.warehouse.dir")

In [ ]:
spark.catalog.currentCatalog()

In [ ]:
df2 = df1.where("age is not null")
df2.show()

In [ ]:
spark.catalog.currentDatabase()

In [ ]:
df2.write.format("json").saveAsTable("users_json")

In [ ]:
spark.catalog.listTables()

In [ ]:
spark.sql("select * from users_json where age > 20").show()

In [ ]:
df2.write.format("csv").saveAsTable("users_csv")

In [ ]:
df2.printSchema()

In [ ]:
spark.sql("select * from users_csv where age > 20").show()

In [ ]:
df2.write.format("csv").option("header", True).option("sep", "\t").saveAsTable("users_tab")

In [ ]:
spark.sql("select count(*) from users_tab where age > 20").show()

In [ ]:
df2.write.format("csv").mode("append").option("header", True).option("sep", "\t").saveAsTable("users_tab")

In [ ]:
### UPDATE, DELETE .. are not supported

#spark.sql("update users_tab set age = age+1 where age is not null")
#spark.sql("delete from users_tab where userid = 1")

### External Tables

In [ ]:
spark.catalog.listTables()

In [ ]:
df2.write.saveAsTable("users_def")

In [ ]:
spark.sql("drop table if exists users_tab")

In [ ]:
path = "E:\\PySpark\\external\\users_external"
df2.write.format("csv").option("path", path).saveAsTable("users_external")

In [ ]:
spark.sql("drop table if exists users_external")

In [ ]:
spark.sql("select * from users_external where age > 15").show()

### Partitioned Tables

In [ ]:
df1.show()

In [ ]:
df1.write.format("csv").option("header", True).partitionBy("gender").saveAsTable("users_part")

In [ ]:
spark.sql("select * from users_part where gender='Male'").show()

### User created database

In [90]:
spark.sql("create database if not exists demodb")
spark.sql("use demodb")

DataFrame[]

In [94]:
#spark.catalog.setCurrentDatabase("demodb")

In [95]:
spark.catalog.currentDatabase()

'demodb'

In [96]:
df2.write.format("json").saveAsTable("users_json")

In [99]:
spark.catalog.listTables()

[Table(name='users_json', catalog='spark_catalog', namespace=['demodb'], description=None, tableType='MANAGED', isTemporary=False)]

In [101]:
spark.sql("drop database demodb cascade")

DataFrame[]

In [102]:
spark.catalog.setCurrentDatabase("default")

In [106]:
spark.catalog.listTables()

[]

In [105]:
spark.sql("drop table users_part")

DataFrame[]